<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline: Random Forest con API de Kaggle
**Proyecto:** Detección de Deslizamientos | **Universidad EAFIT**

Este notebook establece el rendimiento base (baseline) utilizando descriptores de textura HOG y un clasificador tradicional. Implementa la descarga automática desde Kaggle.

## 1. Configuración de Entorno y Kaggle
Carga tu archivo `kaggle.json` para habilitar la descarga directa.

In [ ]:
import os, sys, h5py, zipfile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# ── CONFIGURACIÓN KAGGLE ──
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Sube tu archivo kaggle.json:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# ── DESCARGA Y EXTRACCIÓN ──
if not os.path.exists('/content/landslide4sense'):
    !kaggle datasets download -d landslide4sense/competition -p /content/
    with zipfile.ZipFile('/content/competition.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/landslide4sense')
    print("✓ Dataset listo en /content/landslide4sense")

## 2. Extracción de Características (HOG)
Extraemos descriptores de forma a partir de las bandas RGB para alimentar al Random Forest.

In [ ]:
DATA_ROOT = Path('/content/landslide4sense')
img_dir = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"
N_SAMPLES = 1200 # Ajustar según capacidad de RAM

files = sorted(list(img_dir.glob("*.h5")))[:N_SAMPLES]
X, y = [], []

for f in tqdm(files, desc="Procesando HOG"):
    mf = mask_dir / f.name
    if not mf.exists(): continue
    
    with h5py.File(f, "r") as hf: 
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mf, "r") as hf: 
        mask = hf[list(hf.keys())[0]][()]
    
    # RGB + Normalización básica para HOG
    rgb = patch[:,:,[5,4,3]]
    rgb = ((rgb - rgb.min()) / (rgb.ptp() + 1e-8) * 255).astype("uint8")
    
    # Extracción HOG (Histogram of Oriented Gradients)
    feats = hog(rgb, pixels_per_cell=(16,16), cells_per_block=(2,2), 
                channel_axis=-1, feature_vector=True)
    
    X.append(feats)
    y.append(int(mask.max() > 0))

X, y = np.array(X), np.array(y)
print(f"✓ Dataset preparado: {X.shape[0]} muestras, {X.shape[1]} características.")

## 3. Entrenamiento y Evaluación
Validación cruzada de 5 pliegues para asegurar la estabilidad del baseline.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

print("Iniciando Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X[train_idx], y[train_idx])
    
    preds = rf.predict(X[val_idx])
    f1 = f1_score(y[val_idx], preds)
    f1_scores.append(f1)
    print(f"Fold {fold+1}: F1-Score = {f1:.4f}")

print(f"\n🚀 Resultado Baseline RF: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")